# Lab 6: Transfer learning, detection and segmentation

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Oct 14, 11:59 PM ET · **Target time:** about 75 minutes

---

## What this lab is for

Every model in this lab was trained by somebody else. That is the point. Three
parts:

1. **Transfer learning**, three ways on the same small dataset: a frozen
   pretrained backbone, the same backbone fine-tuned end to end, and the same
   architecture from scratch. You are reproducing Thursday's measurement and
   adding the condition the lecture did not show.
2. **A pretrained detector.** Faster R-CNN on the course photograph. You sweep
   the score threshold, compute IoU by hand, and find the detection that NMS
   should have removed and did not.
3. **A pretrained segmenter**, of each kind. DeepLabV3 and Mask R-CNN on that
   same photograph, and the pixel counts that show what "semantic" and
   "instance" actually mean.

You will not train a detector or a segmenter. That is deliberate, and it is
also what you would do at work.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **You may work together.** Part 1's frozen run is reproducible to the
  decimal. The from-scratch run is **not**, by about a point either way, and
  the lab explains why when you get there.
- **The understanding is assessed on Quiz 7**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.
An agent can call all three of these models for you. It cannot tell you which
of the two boxes in Part 2 is the wrong one, because that takes looking at the
picture.

## Step 0: Setup

::: {.callout-important}
## Turn on the GPU first
Part 1 trains three networks on 224 by 224 images. On a GPU that is about two
minutes total; on a CPU it is about an hour.

In Colab: **Runtime, Change runtime type, T4 GPU**, then run this cell and
check that it prints `cuda`. Do this before you run anything else.
:::

**Downloads.** CIFAR-10 is 170 MB and the three pretrained models are about
560 MB between them. All of it is cached after the first run.

In [ ]:
import os
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.io import read_image

SEED = 6600
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"torch {torch.__version__} on {DEVICE}")
if DEVICE == "cpu":
    print("NO GPU. Part 1 will take about an hour. Go back and read Step 0.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# The photograph Parts 2 and 3 use, the same one from both lectures.
URLS = [
    "https://raw.githubusercontent.com/bthoughton/dsan-lecture-content/main/"
    "deep-learning/lectures/06-detection-and-segmentation/images/cats-dog.jpg",
    "https://raw.githubusercontent.com/bthoughton/dsan-lecture-content/main/"
    "computer-vision/week-1/images/cats-dog.jpg",
]
IMG_PATH = "cats-dog.jpg"

if not os.path.exists(IMG_PATH):
    for url in URLS:
        try:
            urllib.request.urlretrieve(url, IMG_PATH)
            break
        except Exception as e:
            print(f"  could not fetch {url.split('/')[-3]}: {e}")

image = read_image(IMG_PATH)
print(f"image {tuple(image.shape)} {image.dtype}, "
      f"values {int(image.min())} to {int(image.max())}")
plt.figure(figsize=(8, 4.5))
plt.imshow(image.permute(1, 2, 0))
plt.axis("off"); plt.grid(False)
plt.show()

---

# Part 1 · Transfer learning, three ways

CIFAR-10, but only **5,000 training images**. That is small on purpose: it is
the regime where transfer learning matters. With 50,000 you would get a decent
model from scratch and the comparison would be boring.

## 1a · The input contract

A pretrained model does not accept any image you hand it. It expects the exact
preprocessing its weights were trained with, and getting that wrong is a silent
failure: the model runs and returns confident nonsense.

`weights.transforms()` tells you the contract. Read it before you write
anything.

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18

weights = ResNet18_Weights.DEFAULT
tf_default = weights.transforms()
print(tf_default)
print()
print(f"  resize to      {tf_default.resize_size}")
print(f"  then crop to   {tf_default.crop_size}")
print(f"  mean           {tf_default.mean}")
print(f"  std            {tf_default.std}")

Now the judgement call, and it is the reason this is a TODO rather than one
line of copy and paste.

That default pipeline resizes the short side to 256 and then takes a
**center crop** of 224. For a photograph that is sensible: it is the standard
evaluation crop and it throws away a border that usually holds nothing.

A CIFAR image is 32 by 32 and the object fills it. Crop the border off and you
are throwing away the subject. So you want the part of the contract that
matters, the **224 input size and the ImageNet statistics**, without the crop.

In [ ]:
from torchvision import transforms as T

# TODO: build the transform for this dataset.
#   Three steps, in order:
#     1. T.Resize(224), which turns a square 32x32 image into 224x224
#        directly, with no crop
#     2. T.ToTensor()
#     3. T.Normalize(...) with the mean and std you printed above. Read them
#        off tf_default rather than typing the numbers in by hand
tf = None

print(tf)

In [ ]:
train_full = datasets.CIFAR10("data", train=True, download=True, transform=tf)
val_full = datasets.CIFAR10("data", train=False, download=True, transform=tf)
CLASSES = train_full.classes

g = torch.Generator().manual_seed(SEED)
tri = torch.randperm(len(train_full), generator=g)[:5_000].tolist()
vai = torch.randperm(len(val_full), generator=g)[:2_000].tolist()

train_loader = DataLoader(Subset(train_full, tri), batch_size=64, shuffle=True,
                          num_workers=2, generator=g)
val_loader = DataLoader(Subset(val_full, vai), batch_size=64, num_workers=2)

print(f"train {len(tri):,}   val {len(vai):,}   classes {CLASSES}")

## 1b · Three models, one function

All three conditions are the same architecture. What differs is where the
weights start and which of them are allowed to move.

| Condition | Weights start at | What trains |
|---|---|---|
| **Frozen** | ImageNet | the new head only |
| **Fine-tuned** | ImageNet | everything |
| **Scratch** | random | everything |

Replacing the head is the one structural change. ImageNet has 1,000 classes
and CIFAR-10 has 10, so `model.fc` has to be swapped for a `Linear` with 10
outputs. The new layer is randomly initialized and always trainable, which is
why freezing has to happen **before** you replace it.

In [ ]:
def build(pretrained, freeze):
    """ResNet-18 with a 10-class head. Returns the model on DEVICE."""
    torch.manual_seed(SEED)
    model = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)

    # TODO: two steps.
    #   1. If freeze is True, set requires_grad = False on every parameter.
    #      Do this BEFORE the next step, or you will freeze the new head too
    #      and the model will not be able to learn anything.
    #   2. Replace model.fc with an nn.Linear that has the same number of
    #      input features and 10 outputs. model.fc.in_features gives you the
    #      first number.
    raise NotImplementedError("build")

    return model.to(DEVICE)


for pre, frz, name in [(True, True, "frozen"), (True, False, "fine-tune"),
                       (False, False, "scratch")]:
    m = build(pre, frz)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {name:10s} {n:>12,} trainable of "
          f"{sum(p.numel() for p in m.parameters()):,}")

**5,130 against 11,181,642.** The frozen model trains the head and nothing
else, which is $10 \times (512 + 1)$ parameters, about one two-thousandth of
the network. Keep that number in mind when you read the result.

## 1c · Train all three

One detail that is not a magic number. **Fine-tuning gets a smaller learning
rate**, $10^{-4}$ instead of $10^{-3}$. The pretrained features are already
good, and a large step wrecks them before the head is able to make use of
them. The frozen run does not have that problem, because the features cannot
move at all, and the scratch run does not either, because there is nothing
good to wreck.

About two minutes total on a T4.

In [ ]:
def train(model, epochs=8, lr=1e-3):
    train_me = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(train_me, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss()

    hist, t0 = [], time.time()
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            lossf(model(x), y).backward()
            opt.step()
        sched.step()

        model.eval()
        correct = n = 0
        with torch.no_grad():
            for x, y in val_loader:
                correct += (model(x.to(DEVICE)).argmax(1).cpu() == y).sum().item()
                n += len(y)
        hist.append(100 * correct / n)
    return hist, sum(p.numel() for p in train_me), time.time() - t0


runs = {}
for name, pre, frz, lr in [("frozen", True, True, 1e-3),
                           ("fine-tuned", True, False, 1e-4),
                           ("scratch", False, False, 1e-3)]:
    hist, ntr, secs = train(build(pre, frz), lr=lr)
    runs[name] = (hist, ntr)
    print(f"  {name:11s} lr={lr:<6g} {ntr:>11,} trainable   "
          f"epoch 1 {hist[0]:5.2f}   final {hist[-1]:5.2f}   ({secs:.0f}s)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cols = {"frozen": "#1c7a3e", "fine-tuned": "#2980b9", "scratch": "#7f8c8d"}
for name, (hist, ntr) in runs.items():
    ax.plot(range(1, len(hist) + 1), hist, "-o", ms=6, lw=2.6,
            color=cols[name], label=f"{name}  ({ntr:,} trainable)")
ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)")
ax.set_xticks(range(1, len(runs["frozen"][0]) + 1))
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

f, t, s = (runs[k][0][-1] for k in ("frozen", "fine-tuned", "scratch"))
print(f"  fine-tuned beats frozen by  {t - f:+.2f} points")
print(f"  frozen beats scratch by     {f - s:+.2f} points")
print(f"  frozen after ONE epoch ({runs['frozen'][0][0]:.2f}%) against "
      f"scratch after eight ({s:.2f}%)")

## 1d · What the three curves say

**Frozen beats scratch by about eleven points while training 2,180 times fewer
parameters.** The backbone never updates. All it does is turn an image into 512
numbers, and those 512 numbers are good enough that a single linear layer on
top of them beats a whole network trained on this data.

**One epoch of the frozen model beats eight epochs from scratch.** Almost
everything the scratch model is struggling to learn, edges and textures and
parts, is generic, and the pretrained backbone already has it.

**Fine-tuning beats frozen by about twelve more points.** Freezing says the
ImageNet features are exactly right for CIFAR-10. They are close, not exact,
and letting them move the small distance from "ImageNet features" to
"CIFAR-10 features" is worth more than the head alone can buy.

So why ever freeze? Speed, memory, and small data. Freezing trains in half the
time, stores no gradients for the backbone, and cannot overfit the backbone
because the backbone does not move. With 500 images instead of 5,000 the
ordering can reverse.

::: {.callout-note}
## Your scratch number will not match mine
Frozen and fine-tuned reproduce to the decimal. The scratch run does not,
because it is the only one whose every layer is accumulating gradients through
nondeterministic cuDNN kernels. Across six runs it has landed between 64.85 and
66.30. The eleven point gap is the result; the second decimal is not.
:::

---

# Part 2 · A pretrained detector

A classifier answers "what is in this image" with one label. A detector answers
"what is in this image, and where, and how many", and it can say "nothing".
That is a different output shape and a different problem.

## 2a · Run Faster R-CNN

The detector returns three parallel tensors per image: `boxes` as
$(x_1, y_1, x_2, y_2)$ in pixels, `labels` as integer class indices, and
`scores` in $[0, 1]$.

Note what the preprocessing is here. `weights.transforms()` for a detection
model does **not** standardize with the ImageNet constants; it hands the model
plain $[0, 1]$ values and the model standardizes internally. This is the slide
from Thursday about not hard-coding preprocessing, and it is why you ask the
weights rather than assuming.

In [ ]:
from torchvision.models.detection import (fasterrcnn_resnet50_fpn,
                                          FasterRCNN_ResNet50_FPN_Weights)

det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
det_tf = det_weights.transforms()
COCO = det_weights.meta["categories"]

print(f"detector preprocessing mean: {getattr(det_tf, 'mean', None)}")
print(f"detector preprocessing std : {getattr(det_tf, 'std', None)}")
print(f"COCO classes: {len(COCO)}")

detector = fasterrcnn_resnet50_fpn(weights=det_weights).eval().to(DEVICE)
with torch.no_grad():
    out = detector([det_tf(image).to(DEVICE)])[0]

boxes = out["boxes"].cpu()
labels = [COCO[i] for i in out["labels"].cpu()]
scores = out["scores"].cpu()
print(f"\nthe model returned {len(boxes)} detections")
for b, l, s in zip(boxes[:8], labels[:8], scores[:8]):
    print(f"  {l:14s} {s:.3f}  [{b[0]:7.0f} {b[1]:7.0f} {b[2]:7.0f} {b[3]:7.0f}]")

Eighteen detections, and the photograph has three animals in it. The model has
already run non-maximum suppression internally; these eighteen are what
survived it. Everything below is about what you do with them.

## 2b · The score threshold is yours to choose

The model does not decide how many objects are in the image. **You** do, when
you pick a threshold. Nothing about the model changes as you sweep it.

In [ ]:
# TODO: count how many detections survive at each threshold.
#   scores is a tensor; (scores >= t) gives a boolean tensor and .sum()
#   counts the True values.

for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    kept = None
    print(f"  threshold {t:.1f}  ->  {kept} detections")

In [ ]:
def draw(ax, bxs, lbs, scs, title):
    ax.imshow(image.permute(1, 2, 0))
    for b, l, s in zip(bxs, lbs, scs):
        ax.add_patch(patches.Rectangle(
            (b[0], b[1]), b[2] - b[0], b[3] - b[1],
            fill=False, color="#1c7a3e", lw=2.2))
        ax.text(b[0], b[1] - 8, f"{l} {s:.2f}", fontsize=8, color="white",
                bbox=dict(facecolor="#1c7a3e", pad=1, edgecolor="none"))
    ax.set_title(title, fontsize=10)
    ax.axis("off"); ax.grid(False)


fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
for ax, t in zip(axes, (0.3, 0.5, 0.7)):
    k = scores >= t
    draw(ax, boxes[k], [l for l, m in zip(labels, k) if m], scores[k],
         f"threshold {t}: {int(k.sum())} detections")
plt.tight_layout()
plt.show()

At 0.3 you get a box that is not there. At 0.7 you lose one that is. There is
no setting that is correct in general, only one that is correct for what the
mistake costs you. A medical screening tool and a photo-tagging feature want
very different thresholds from the same model.

## 2c · IoU, by hand

**Intersection over Union** measures how much two boxes agree:

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

It is 0 for boxes that do not touch and 1 for identical boxes. Every number in
detection, from NMS to mAP, is built on it.

In [ ]:
def iou(a, b):
    """IoU of two boxes, each (x1, y1, x2, y2)."""
    # TODO: four steps.
    #   1. The intersection rectangle: its left edge is the LARGER of the two
    #      left edges, its right edge is the SMALLER of the two right edges,
    #      and likewise for top and bottom.
    #   2. Its width and height, each clamped at 0: boxes that do not overlap
    #      give a negative width, and a negative area is not what you want.
    #   3. Each box's own area.
    #   4. intersection / (areaA + areaB - intersection).
    raise NotImplementedError("iou")


# Two boxes that do not touch, and two that are identical.
print(f"  disjoint  {iou(torch.tensor([0., 0., 10., 10.]), torch.tensor([20., 20., 30., 30.])):.3f}")
print(f"  identical {iou(torch.tensor([0., 0., 10., 10.]), torch.tensor([0., 0., 10., 10.])):.3f}")
print(f"  half over {iou(torch.tensor([0., 0., 10., 10.]), torch.tensor([5., 0., 15., 10.])):.3f}")

Now run it on two real detections. Keep the detections above 0.5 and compare
every pair.

In [ ]:
keep = scores >= 0.5
kb, kl, ks = boxes[keep], [l for l, m in zip(labels, keep) if m], scores[keep]

print(f"  {len(kb)} detections above 0.5:")
for i, (l, s) in enumerate(zip(kl, ks)):
    print(f"    {i}  {l:6s} {s:.3f}")

print("\n  pairwise IoU:")
worst = (0.0, None)
for i in range(len(kb)):
    for j in range(i + 1, len(kb)):
        v = float(iou(kb[i], kb[j]))
        flag = ""
        if v > 0.5:
            flag = "  <-- these two boxes are nearly the same box"
            if v > worst[0]:
                worst = (v, (i, j))
        print(f"    {kl[i]:6s} {i} and {kl[j]:6s} {j}:  {v:.3f}{flag}")

## 2d · The duplicate that survived

One pair has an IoU above 0.8. Those are two boxes around **the same animal**,
and the detector kept both.

Non-maximum suppression is supposed to prevent exactly this. It sorts by score,
keeps the best box, and throws away anything overlapping it by more than some
IoU. It ran, and it did not remove this pair, because torchvision runs NMS
**per class**: it compares dog boxes to dog boxes and cat boxes to cat boxes,
and never compares a dog box to a cat box.

So a confident dog and a less confident cat drawn around the same animal both
survive. Watch what changes when you ignore the labels.

In [ ]:
from torchvision.ops import nms, batched_nms

# TODO: run NMS two ways on the detections above 0.5, both with iou_threshold
#   0.5, and count what survives each.
#     per_class:  batched_nms(boxes, scores, class_indices, iou_threshold)
#                 compares boxes only against others with the same index
#     agnostic:   nms(boxes, scores, iou_threshold)
#                 compares every box against every other box
#   class_idx is given to you below.

class_idx = out["labels"].cpu()[keep]

per_class = None
agnostic = None

print(f"  started with            {len(kb)}")
print(f"  after per-class NMS     {len(per_class)}")
print(f"  after class-agnostic NMS {len(agnostic)}")

In [ ]:
removed = sorted(set(range(len(kb))) - set(agnostic.tolist()))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
draw(axes[0], kb, kl, ks, f"per-class NMS: {len(per_class)} boxes")
ai = agnostic.tolist()
draw(axes[1], kb[ai], [kl[i] for i in ai], ks[ai],
     f"class-agnostic NMS: {len(agnostic)} boxes")
plt.tight_layout()
plt.show()

for i in removed:
    print(f"  class-agnostic NMS removed: {kl[i]} at {ks[i]:.3f}")
print("\n  Look at the two pictures. Which one is right?")

Class-agnostic NMS removes the duplicate. It would also remove a real cat
genuinely sitting in front of a real dog, which is why it is not the default.
Neither choice is free, and neither is a setting you can leave alone and forget
about.

**This is the honest summary of Part 2.** The detector did not give you an
answer. It gave you eighteen scored guesses, and three separate decisions were
yours: the score threshold, the NMS IoU threshold, and whether NMS compares
across classes. Every one of them changes how many objects you report.

---

# Part 3 · Two kinds of segmentation

A box says roughly where. A **mask** says exactly which pixels. There are two
kinds and the difference is not a detail.

- **Semantic segmentation** labels every pixel with a class. Two cats touching
  are one region of "cat".
- **Instance segmentation** returns one mask per object. Two cats are two
  masks, and they are numbered.

Same photograph, one of each.

## 3a · Semantic: DeepLabV3

In [ ]:
from torchvision.models.segmentation import (deeplabv3_resnet50,
                                             DeepLabV3_ResNet50_Weights)

seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
VOC = seg_weights.meta["categories"]
seg_tf = seg_weights.transforms()

segmenter = deeplabv3_resnet50(weights=seg_weights).eval().to(DEVICE)
with torch.no_grad():
    logits = segmenter(seg_tf(image).unsqueeze(0).to(DEVICE))["out"][0]

sem = logits.argmax(0).cpu()
print(f"semantic map {tuple(sem.shape)}, one class index per pixel")
print(f"classes present: {[VOC[i] for i in sem.unique().tolist()]}")

In [ ]:
# TODO: for every class the map found other than the background, print how
#   many pixels carry it and what share of the image that is.
#   (sem == k) is a boolean map; .sum() counts it. VOC[0] is "__background__".

total = sem.numel()
for k in sem.unique().tolist():
    if VOC[k] == "__background__":
        continue
    px = None
    print(f"  {VOC[k]:10s} {px:>8,} px   {100 * px / total:5.1f}% of the image")

Two classes, cat and dog, and **the two cats are one region**. There is no
count in this output and no way to get one: a semantic model cannot tell you
how many cats there are, only which pixels are cat. If you need a count, this
is the wrong model.

Also worth noticing: `bed` is not in the output, because it is not one of the
21 Pascal VOC classes this model was trained on. The detector found a bed
because COCO has one. A pretrained model can only ever return its own
vocabulary.

## 3b · Instance: Mask R-CNN

In [ ]:
from torchvision.models.detection import (maskrcnn_resnet50_fpn,
                                          MaskRCNN_ResNet50_FPN_Weights)

mask_weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
masker = maskrcnn_resnet50_fpn(weights=mask_weights).eval().to(DEVICE)
with torch.no_grad():
    mout = masker([mask_weights.transforms()(image).to(DEVICE)])[0]

mkeep = mout["scores"].cpu() >= 0.5
mlabels = [mask_weights.meta["categories"][i]
           for i, m in zip(mout["labels"].cpu(), mkeep) if m]
mscores = mout["scores"].cpu()[mkeep]
masks = mout["masks"].cpu()[mkeep]
print(f"instance masks {tuple(masks.shape)}, one per object, soft in [0, 1]")

In [ ]:
# TODO: a Mask R-CNN mask is soft, a probability per pixel. Threshold each one
#   at 0.5 to make it a hard mask, then count its pixels.
#   masks has shape (n, 1, H, W), so masks[:, 0] drops the singleton axis.
#   Store the thresholded masks in `hard`, which the figure below uses.

hard = None

print(f"  {len(masks)} instances:")
for i in range(len(masks)):
    px = None
    print(f"    {i}  {mlabels[i]:6s} {mscores[i]:.3f}   {px:>8,} px")

In [ ]:
fig, axes = plt.subplots(1, len(masks) + 1, figsize=(3.1 * (len(masks) + 1), 2.6))
axes[0].imshow(sem, cmap="tab20")
axes[0].set_title(f"semantic: {len(sem.unique()) - 1} classes", fontsize=9)
for ax, i in zip(axes[1:], range(len(masks))):
    ax.imshow(image.permute(1, 2, 0))
    ax.imshow(np.ma.masked_where(~hard[i], hard[i]), alpha=0.6, cmap="autumn")
    ax.set_title(f"{mlabels[i]} {mscores[i]:.2f}", fontsize=9)
for ax in axes:
    ax.axis("off"); ax.grid(False)
plt.tight_layout()
plt.show()

## 3c · The difference, stated plainly

The semantic model returned **one class called cat**, covering 17.2% of the
pixels. It did not say how many cats that is, and it has no way to. The
instance model returned **two cats**, numbered, each with its own mask and its
own score.

The two cats happen not to touch in this photograph, so you could recover the
count by finding connected regions yourself. Slide them together and that trick
fails, while Mask R-CNN still returns two.

Neither is more correct. They answer different questions, and which one you
want is decided by whether you need to count:

- "What fraction of this field is diseased?" is semantic. Counting plants would
  not help you.
- "How many cars are in this lot?" is instance. A single region of "car" is
  useless.

The instance model also found the bed, which the semantic model could not,
because of vocabulary rather than architecture.

---

# Part 4 · What this lab showed

Eight questions. If you can answer them, you are ready for Quiz 7.

1. **Why a frozen backbone with 5,130 trainable parameters beat a whole
   network trained from scratch** on the same 5,000 images.
2. **Why fine-tuning beat freezing**, and the two or three situations where you
   would still freeze.
3. **Why fine-tuning used a smaller learning rate** than the other two runs.
4. **What `weights.transforms()` is for**, and why you did not hard-code the
   ImageNet constants. What did the detector's preprocessing turn out to be?
5. **Who decides how many objects are in the image**, and the three separate
   knobs in Part 2 that changed the answer.
6. **What IoU is**, and what it is 0 and 1 for.
7. **Why NMS left two boxes around the same animal**, and what removing them
   would cost you on a different photograph.
8. **The difference between semantic and instance segmentation**, said in terms
   of a question you could answer with one and not the other.

---

# Part 5 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that every plot and every printed block is visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Oct 14, 11:59 PM ET**.

The notebook is graded for completion, so this is mostly a formality. Part 4 is
your study guide for the quiz that assesses the understanding.

::: {.callout-note}
## Do not submit the data folder
Running this notebook creates a `data/` directory of about 170 MB next to it,
plus the cached model weights. Submit the rendered HTML or PDF, not the folder.
:::